In [1]:
# STEP 1 — Imports, paths, and load EPC dataset
# Purpose: Load EPC data and define file locations for building lighting schedules

import pandas as pd
from pathlib import Path

# Path to EPC dataset
epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"

# Base path to building models
base_models_path = Path("/Users/jakegehrung/Desktop/data_raw/baseline_models")

# Load EPC data
epc_df = pd.read_csv(epc_path)

# Quick check
epc_df.head()

,BUILDING_REFERENCE_NUMBER,OSG_REFERENCE_NUMBER,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,LATITUDE,LONGITUDE,ORIENTATION,ORIENTATION_ESPR_ROT,...,PV_NPV_FLEX,BATTERY_NPV_FLEX,WINDOWS_IRR_FLEX,ROOF_IRR_FLEX,WALLS_IRR_FLEX,FAB_IRR_FLEX,HP_IRR_FLEX,SOLAR_THERMAL_IRR_FLEX,PV_IRR_FLEX,BATTERY_IRR_FLEX
0,1001829067,122009933.0,19 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.0557,-4.2233,100,170,...,0.000000,-4836.000000,NaN,NaN,NaN,NaN,0.227522,0.105684,NaN,NaN
1,1001951858,122010025.0,GLENVIEW,FINTRY,GLASGOW,G63 0XJ,56.0528,-4.2206,180,90,...,1962.148428,-2558.147057,-0.015884,-0.074175,-0.017873,-0.009717,0.418858,0.137610,0.081951,-0.037370
2,1001829071,122009868.0,22 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.0555,-4.2237,100,170,...,-2000.000000,-4836.000000,0.115626,1.741603,NaN,NaN,NaN,-0.040909,NaN,NaN
3,1234761001,122009968.0,1 MENZIES TERRACE,FINTRY,GLASGOW,G63 0YJ,56.0584,-4.2248,150,120,...,1179.292669,-2385.903238,-0.079761,-0.058603,-0.003530,0.003958,-0.012701,0.006844,0.073403,-0.029204
4,1001991633,122009892.0,50 MAIN STREET,FINTRY,GLASGOW,G63 0XF,56.0547,-4.2283,150,120,...,-21000.000000,-4836.000000,0.372136,2.422694,NaN,NaN,NaN,0.129009,NaN,NaN


In [2]:
# STEP 2 — Create mapping from BUILDING_REFERENCE_NUMBER to LIGHTING_WATTS
# Purpose: Enable fast lookup of lighting power per building

epc_df["BUILDING_REFERENCE_NUMBER"] = epc_df["BUILDING_REFERENCE_NUMBER"].astype(str)

lighting_map = dict(
    zip(epc_df["BUILDING_REFERENCE_NUMBER"], epc_df["LIGHTING_WATTS"])
)

# Sanity check
len(lighting_map)

117

In [3]:
# STEP 3 — Helper function to process a single building lighting schedule
# Purpose: Read schedule, compute lighting power, format Time column, return final dataframe

import pandas as pd

def process_lighting_schedule(building_id, lighting_watts, base_models_path):
    """
    Reads lighting_schedule.csv for a building and returns formatted output.
    """

    lighting_folder = base_models_path / str(building_id) / "LIGHTING"
    schedule_path = lighting_folder / "lighting_schedule.csv"

    if not schedule_path.exists():
        print(f"Missing schedule for building {building_id}")
        return None

    df = pd.read_csv(schedule_path)

    # Identify time column or index
    time_col = None
    for col in df.columns:
        if col.lower() in ["time", "timestamp", "datetime", "date_time", "date"]:
            time_col = col
            break

    if time_col is not None:
        # Convert to datetime then extract HH:MM
        df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
        df["Time"] = df[time_col].dt.strftime("%H:%M:%S")
    else:
        # Assume first column is time-like or index represents time
        df = df.reset_index()
        df["Time"] = pd.to_datetime(df.iloc[:, 0], errors="coerce").dt.strftime("%H:%M:%S")

    # Identify lighting fraction column
    fraction_col = None
    for col in df.columns:
        if "fraction" in col.lower():
            fraction_col = col
            break

    if fraction_col is None:
        raise ValueError(f"No lighting_fraction column found for building {building_id}")

    # Compute lighting power
    df["lighting_power"] = lighting_watts * df[fraction_col]

    # Final output
    output_df = df[["Time", "lighting_power"]].copy()

    return output_df

In [4]:
# STEP 4 — Loop through all buildings and generate lighting_power.csv
# Purpose: Process each building and save output inside its LIGHTING folder

for building_id, lighting_watts in lighting_map.items():

    lighting_folder = base_models_path / str(building_id) / "LIGHTING"

    try:
        result_df = process_lighting_schedule(building_id, lighting_watts, base_models_path)

        if result_df is None:
            continue

        output_path = lighting_folder / "lighting_power.csv"

        result_df.to_csv(output_path, index=False)

        print(f"Saved: {output_path}")

    except Exception as e:
        print(f"Error processing {building_id}: {e}")

/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")


Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829067/LIGHTING/lighting_power.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001951858/LIGHTING/lighting_power.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829071/LIGHTING/lighting_power.csv


/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")


Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1234761001/LIGHTING/lighting_power.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001991633/LIGHTING/lighting_power.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664929/LIGHTING/lighting_power.csv


/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")


...

Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1002031280/LIGHTING/lighting_power.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829074/LIGHTING/lighting_power.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829065/LIGHTING/lighting_power.csv


/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")


Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001063643/LIGHTING/lighting_power.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001063635/LIGHTING/lighting_power.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829066/LIGHTING/lighting_power.csv


/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_34718/737393887.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], errors="coerce")


Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829061/LIGHTING/lighting_power.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627540/LIGHTING/lighting_power.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627570/LIGHTING/lighting_power.csv


In [5]:
# STEP 5 — Validation check (optional)
# Purpose: Confirm a sample output file structure is correct

sample_building = next(iter(lighting_map.keys()))
sample_path = base_models_path / str(sample_building) / "LIGHTING" / "lighting_power.csv"

if sample_path.exists():
    sample_df = pd.read_csv(sample_path)
    display(sample_df.head())
else:
    print("Sample output not found.")

,Time,lighting_power
0,00:30:00,12.066229
1,01:00:00,8.563129
2,01:30:00,6.227729
3,02:00:00,5.060029
4,02:30:00,4.281571
